# Finite width and actual factor gradient descent

We sample neuron columns, then update both factor blocks simultaneously. The covariance of a factor step has a quadratic term in the stepsize and is not an explicit Euler step of the covariance ODE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from linear_networks import (
    QuadraticLoss, SmoothedLassoLoss, sample_factors, factor_covariance,
    factor_step, covariance_step, covariance_rhs, force_matrix,
    gradient_descent, integrate_covariance, signed_spectrum, support_margins,
)

loss = QuadraticLoss([[0.8, -0.4], [0.2, 0.6]])
d_u, width = 2, 16
population = 0.6**2 * np.eye(4)
w0 = sample_factors(population, width, seed=7)
sigma0 = factor_covariance(w0)
print(f'Population gap: {support_margins(population, d_u).gap:.4f}')
print(f'Empirical gap: {support_margins(sigma0, d_u).gap:.4f}')

## Check one step

With $G=G_{\nabla\ell(\Sigma^{vu})}$, factor GD induces
$$\Sigma_+=(I-\tau G)\Sigma(I-\tau G)
=\Sigma+\tau\dot\Sigma+\tau^2G\Sigma G.$$
The same pre-update gradient must be used for both $U$ and $V$.

In [ ]:
tau = 0.025
actual = factor_covariance(factor_step(w0, loss, tau))
closed = covariance_step(sigma0, loss, tau)
euler = sigma0 + tau * covariance_rhs(sigma0, loss)
g = force_matrix(loss.gradient(sigma0[d_u:, :d_u]))
np.testing.assert_allclose(actual, closed, atol=1e-14)
np.testing.assert_allclose(closed - euler, tau**2 * g @ sigma0 @ g, atol=1e-14)
assert np.linalg.norm(closed - euler) > 1e-7

## Compare descent with gradient flow

The chosen step is illustrative, not a universal certified stepsize. We check this particular run's loss decreases. The continuous flow preserves the signed spectrum, whereas discrete GD generally drifts.

In [ ]:
gd = gradient_descent(w0, loss, step_size=tau, steps=300)
gf = integrate_covariance(sigma0, loss, gd.times)
assert np.max(np.diff(gd.loss_gaps)) < 1e-10
initial_spectrum = signed_spectrum(sigma0, d_u)
drift = np.array([np.linalg.norm(signed_spectrum(s, d_u) - initial_spectrum) for s in gd.covariances])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), constrained_layout=True)
axes[0].semilogy(gd.times, gd.loss_gaps, label='factor GD', color='#2458A6')
axes[0].semilogy(gf.times, gf.loss_gaps, '--', label='covariance gradient flow', color='#D1495B')
axes[0].set(xlabel='effective time k tau', ylabel='loss gap')
axes[0].legend()
axes[1].plot(gd.times, drift, color='#00876C')
axes[1].set(xlabel='effective time k tau', ylabel='signed-spectrum drift under GD')
plt.show()

## Swap the predictor loss

The dynamics are not restricted to quadratic losses. Smoothed Lasso has a regularized optimum that differs from the teacher, so its loss gap is measured from that optimum.

In [ ]:
regularized = SmoothedLassoLoss(loss.target, strength=0.1, smoothing=0.05)
regularized_path = gradient_descent(w0, regularized, step_size=0.01, steps=400)
assert np.max(np.diff(regularized_path.loss_gaps)) < 1e-10
print('Regularized optimum:')
print(regularized.minimizer)
print(f'Final regularized loss gap: {regularized_path.loss_gaps[-1]:.3e}')